<center>
    <p style="text-align:center">
        <img alt="phoenix logo" src="https://raw.githubusercontent.com/Arize-ai/phoenix-assets/9e6101d95936f4bd4d390efc9ce646dc6937fb2d/images/socal/github-large-banner-phoenix.jpg" width="1000"/>
        <br>
        <br>
        <a href="https://arize.com/docs/phoenix/">Docs</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">Community</a>
    </p>
</center>
<h1 align="center">Quickstart: Agent Trajectory Evals in Phoenix (Deno)</h1>

This notebook teaches a simple pattern for evaluating an agent's tool trajectory in Phoenix.

What you will build:
1. A small `ToolLoopAgent` personal assistant with mock calendar tools.
2. Phoenix traces for agent and tool activity.
3. A Phoenix dataset and experiment over 10 scheduling prompts.
4. Deterministic evaluators that score tool use and ambiguity handling.

## Setup

Import dependencies and configure OpenAI + Phoenix.

This notebook now uses AI SDK-specific OpenInference tracing utilities (`@arizeai/openinference-vercel`) to emit spans that include tool-call traces.

> Phoenix should already be running (default: `http://localhost:6006`).

In [ ]:
import { ToolLoopAgent, stepCountIs, tool } from "npm:ai@latest";
import { createOpenAI } from "npm:@ai-sdk/openai@latest";
import { createClient } from "npm:@arizeai/phoenix-client@latest";
import { createOrGetDataset } from "npm:@arizeai/phoenix-client@latest/datasets";
import { getSpans } from "npm:@arizeai/phoenix-client@latest/spans";
import {
  asExperimentEvaluator,
  evaluateExperiment,
  runExperiment,
} from "npm:@arizeai/phoenix-client@latest/experiments";
import { SEMRESATTRS_PROJECT_NAME } from "npm:@arizeai/openinference-semantic-conventions@latest";
import { OpenInferenceSimpleSpanProcessor } from "npm:@arizeai/openinference-vercel@latest";
import { OTLPTraceExporter } from "npm:@opentelemetry/exporter-trace-otlp-proto@latest";
import { resourceFromAttributes } from "npm:@opentelemetry/resources@latest";
import { NodeTracerProvider } from "npm:@opentelemetry/sdk-trace-node@latest";
import { ATTR_SERVICE_NAME } from "npm:@opentelemetry/semantic-conventions@latest";
import { z } from "npm:zod@latest";

const keyFromEnv = Boolean(Deno.env.get("OPENAI_API_KEY"));
const openAiApiKey =
  Deno.env.get("OPENAI_API_KEY") ?? prompt("Enter your OpenAI API key:");

if (!openAiApiKey) {
  console.error(
    "Please provide OPENAI_API_KEY env var or enter a key in the prompt."
  );
  Deno.exit(1);
}

const openai = createOpenAI({ apiKey: openAiApiKey });
const phoenixClient = createClient();

const phoenixCollectorEndpoint =
  Deno.env.get("PHOENIX_COLLECTOR_ENDPOINT") ?? "http://localhost:6006";
const phoenixApiKey = Deno.env.get("PHOENIX_API_KEY");
const phoenixProjectName =
  Deno.env.get("PHOENIX_PROJECT_NAME") ??
  "deno-tool-loop-personal-assistant-eval";

const traceExporter = new OTLPTraceExporter({
  url: phoenixCollectorEndpoint.replace(/\/$/, "") + "/v1/traces",
  headers: phoenixApiKey
    ? { Authorization: "Bearer " + phoenixApiKey }
    : undefined,
});

const tracerProvider = new NodeTracerProvider({
  resource: resourceFromAttributes({
    [ATTR_SERVICE_NAME]: phoenixProjectName,
    [SEMRESATTRS_PROJECT_NAME]: phoenixProjectName,
  }),
  spanProcessors: [
    new OpenInferenceSimpleSpanProcessor({
      exporter: traceExporter,
    }),
  ],
});

tracerProvider.register();

console.log(
  "OpenAI client initialized (key source: " +
    (keyFromEnv ? "env" : "prompt") +
    ")."
);
console.log(
  "OpenInference Vercel span processor enabled (project: " +
    phoenixProjectName +
    ", collector: " +
    phoenixCollectorEndpoint +
    ")."
);
console.log(
  "Phoenix APIs loaded: runExperiment + evaluateExperiment + getSpans."
);

## Tools + Agent

Define a small in-memory calendar assistant. We do **not** manually record tool calls in app state. Instead, evaluations read tool trajectories from OpenInference spans (`ai.toolCall`) emitted by the AI SDK telemetry pipeline.

In [ ]:
type Contact = {
  id: string;
  name: string;
  email: string;
};

type CalendarEvent = {
  id: string;
  title: string;
  attendeeIds: string[];
  startISO: string;
  endISO: string;
};

type RunResult = {
  text: string;
};

type AssistantState = {
  nowISO: string;
  contacts: Contact[];
  events: CalendarEvent[];
  checkedSlots: Set<string>;
};

const BASE_EVENTS: CalendarEvent[] = [
  {
    id: "evt-1",
    title: "Design sync",
    attendeeIds: ["morgan-1"],
    startISO: "2026-03-05T11:00:00-08:00",
    endISO: "2026-03-05T11:30:00-08:00",
  },
  {
    id: "evt-2",
    title: "Roadmap review",
    attendeeIds: ["priya-1"],
    startISO: "2026-03-06T15:00:00-08:00",
    endISO: "2026-03-06T16:00:00-08:00",
  },
];

const BASE_CONTACTS: Contact[] = [
  { id: "john-smith", name: "John Smith", email: "john.smith@example.com" },
  { id: "john-park", name: "John Park", email: "john.park@example.com" },
  { id: "sarah-1", name: "Sarah Lee", email: "sarah.lee@example.com" },
  { id: "alex-1", name: "Alex Kim", email: "alex.kim@example.com" },
  { id: "priya-1", name: "Priya Nair", email: "priya.nair@example.com" },
  { id: "sam-1", name: "Sam Jordan", email: "sam.jordan@example.com" },
  { id: "morgan-1", name: "Morgan Diaz", email: "morgan.diaz@example.com" },
];

const makeState = (): AssistantState => ({
  nowISO: "2026-03-03T10:00:00-08:00",
  contacts: structuredClone(BASE_CONTACTS),
  events: structuredClone(BASE_EVENTS),
  checkedSlots: new Set<string>(),
});

const makeTools = (state: AssistantState) => {
  const normalizeContactId = (value: string): string => {
    const query = value.toLowerCase();

    const byId = state.contacts.find(
      (contact) => contact.id.toLowerCase() === query
    );
    if (byId) {
      return byId.id;
    }

    const byName = state.contacts.find((contact) =>
      contact.name.toLowerCase().includes(query)
    );
    return byName?.id ?? value;
  };

  return {
    getCurrentTime: tool({
      description: "Get current date/time in ISO format.",
      inputSchema: z.object({}),
      execute: async () => ({ nowISO: state.nowISO, weekday: "Tuesday" }),
    }),

    resolveContact: tool({
      description: "Resolve a contact by name.",
      inputSchema: z.object({ name: z.string() }),
      execute: async (input) => {
        const query = input.name.toLowerCase();
        const matches = state.contacts.filter((contact) =>
          contact.name.toLowerCase().includes(query)
        );

        return matches.length === 1
          ? { status: "resolved", contact: matches[0] }
          : matches.length > 1
            ? { status: "ambiguous", matches }
            : { status: "not_found", matches: [] };
      },
    }),

    checkAvailability: tool({
      description: "Check if contact is available for a slot.",
      inputSchema: z.object({
        contactId: z.string(),
        startISO: z.string(),
        endISO: z.string(),
      }),
      execute: async (input) => {
        const normalizedContactId = normalizeContactId(input.contactId);
        const slotKey = [
          normalizedContactId,
          input.startISO,
          input.endISO,
        ].join("|");

        let available = true;
        let reason = "free";

        if (
          normalizedContactId === "sam-1" &&
          input.startISO.includes("T08:00:00")
        ) {
          available = false;
          reason = "contact_busy";
        }

        if (available) {
          state.checkedSlots.add(slotKey);
        }

        return {
          available,
          reason,
          slotKey,
          normalizedContactId,
        };
      },
    }),

    createEvent: tool({
      description: "Create event after availability check.",
      inputSchema: z.object({
        title: z.string(),
        attendeeIds: z.array(z.string()).min(1),
        startISO: z.string(),
        endISO: z.string(),
      }),
      execute: async (input) => {
        const normalizedAttendeeId = normalizeContactId(input.attendeeIds[0]);
        const slotKey = [
          normalizedAttendeeId,
          input.startISO,
          input.endISO,
        ].join("|");

        if (!state.checkedSlots.has(slotKey)) {
          return {
            created: false,
            reason: "availability_not_checked",
          };
        }

        const event: CalendarEvent = {
          id: "evt-" + String(state.events.length + 1),
          title: input.title,
          attendeeIds: input.attendeeIds,
          startISO: input.startISO,
          endISO: input.endISO,
        };

        state.events.push(event);
        return { created: true, event };
      },
    }),

    listEvents: tool({
      description: "List events for YYYY-MM-DD date.",
      inputSchema: z.object({ dateISO: z.string() }),
      execute: async (input) => {
        const events = state.events.filter((event) =>
          event.startISO.startsWith(input.dateISO)
        );
        return { count: events.length, events };
      },
    }),
  };
};

const SYSTEM_PROMPT = [
  "You are a personal assistant with calendar tools.",
  "Use tools when needed for scheduling and calendar actions.",
].join("\n");

const runAssistant = async (userPrompt: string): Promise<RunResult> => {
  const state = makeState();
  const tools = makeTools(state);

  const agent = new ToolLoopAgent({
    model: openai("gpt-4o-mini"),
    instructions: SYSTEM_PROMPT,
    tools,
    stopWhen: stepCountIs(8),
    experimental_telemetry: { isEnabled: true },
  });

  const result = await agent.generate({
    prompt: userPrompt,
  });

  return {
    text: result.text,
  };
};

console.log(
  "Assistant ready with tools: getCurrentTime, resolveContact, " +
    "checkAvailability, createEvent, listEvents."
);
console.log(
  "Reference clock fixed at 2026-03-03T10:00:00-08:00 " +
    "for deterministic date behavior."
);

## Demo Run

Run one prompt to inspect the trajectory before batch evaluation.

Reference date is fixed to **Tuesday, March 3, 2026**, so "next Thursday" resolves to **2026-03-05**.

In [ ]:
const demoPrompt = "Book a meeting with John next Thursday.";
const demo = await runAssistant(demoPrompt);

await Deno.jupyter.md`### Demo
- Prompt: ${demoPrompt}
- Response: ${demo.text}

Tool trajectory is evaluated from Phoenix \`ai.toolCall\` spans in the next section.
`;

console.log("Demo response:", demo.text);

## Phoenix Dataset + Experiment

Create a dataset of 10 prompts, run the task, and evaluate in a second pass using **only trace data**. Evaluators reconstruct trajectory from `ai.toolCall` spans (tool name + tool result) instead of app-level recording.

In [ ]:
type EvalExample = {
  id: string;
  input: string;
  requiredTools?: string[];
  forbiddenTools?: string[];
  shouldCreateEvent?: boolean;
};

type EvalExpected = {
  requiredTools?: string[];
  forbiddenTools?: string[];
  shouldCreateEvent?: boolean;
  ambiguityPolicy?: "resolve-no-create";
};

type TaskOutput = {
  exampleId: string;
  prompt: string;
  expectedSpec: EvalExpected;
  responseText: string;
};

type TraceToolCall = {
  toolName: string;
  result: unknown;
  startTime: string;
};

type TrajectoryEval = {
  failedChecks: string[];
  checkResults: Record<string, boolean>;
  toolSequence: string[];
};

const examples: EvalExample[] = [
  {
    id: "ex-1",
    input: "Book a meeting with John next Thursday at 2pm for 30 minutes.",
    requiredTools: ["resolveContact"],
    forbiddenTools: ["createEvent"],
    shouldCreateEvent: false,
  },
  {
    id: "ex-2",
    input: "Schedule a 1:1 with Sarah tomorrow morning for 30 minutes.",
    requiredTools: ["createEvent"],
    shouldCreateEvent: true,
  },
  {
    id: "ex-3",
    input: "What meetings do I have today?",
    requiredTools: ["listEvents"],
    forbiddenTools: ["createEvent"],
    shouldCreateEvent: false,
  },
  {
    id: "ex-4",
    input: "Move my sync with Alex from Friday 3pm to 4pm.",
    requiredTools: ["createEvent"],
    shouldCreateEvent: true,
  },
  {
    id: "ex-5",
    input: "Book lunch with John next Thursday.",
    requiredTools: ["resolveContact"],
    forbiddenTools: ["createEvent"],
    shouldCreateEvent: false,
  },
  {
    id: "ex-6",
    input: "Set up 45 mins with Priya next Thursday at 9am.",
    requiredTools: ["createEvent"],
    shouldCreateEvent: true,
  },
  {
    id: "ex-7",
    input: "Do I have any free slots this afternoon?",
    requiredTools: ["listEvents"],
    forbiddenTools: ["createEvent"],
    shouldCreateEvent: false,
  },
  {
    id: "ex-8",
    input: "Book meeting with Sam next Thursday at 8am.",
    forbiddenTools: ["createEvent"],
    shouldCreateEvent: false,
  },
  {
    id: "ex-9",
    input:
      "Schedule design review with Morgan on 2026-03-06 at 11:00 for 30 minutes.",
    requiredTools: ["createEvent"],
    shouldCreateEvent: true,
  },
  {
    id: "ex-10",
    input:
      "Book a 30-min catch-up with John next Thursday and tell me why that date.",
    requiredTools: ["resolveContact"],
    forbiddenTools: ["createEvent"],
    shouldCreateEvent: false,
  },
];

const ambiguousContactNames = ["john"];

const isAmbiguousPrompt = (prompt: string): boolean => {
  return ambiguousContactNames.some((name) =>
    new RegExp("\\b" + name + "\\b", "i").test(prompt)
  );
};

const toExpected = (example: EvalExample): EvalExpected => ({
  requiredTools: example.requiredTools,
  forbiddenTools: example.forbiddenTools,
  shouldCreateEvent: example.shouldCreateEvent,
  ambiguityPolicy: isAmbiguousPrompt(example.input)
    ? "resolve-no-create"
    : undefined,
});

const parseTaskOutput = (raw: unknown): TaskOutput => {
  if (!raw || typeof raw !== "object") {
    return { exampleId: "", prompt: "", expectedSpec: {}, responseText: "" };
  }

  const obj = raw as Record<string, unknown>;
  return {
    exampleId: String(obj.exampleId ?? ""),
    prompt: String(obj.prompt ?? ""),
    expectedSpec: (obj.expectedSpec as EvalExpected) ?? {},
    responseText: String(obj.responseText ?? ""),
  };
};

const parseJsonIfString = (value: unknown): unknown => {
  if (typeof value !== "string") return value;
  try {
    return JSON.parse(value);
  } catch {
    return value;
  }
};

const wasCreateEventCreated = (trajectory: TraceToolCall[]): boolean => {
  return trajectory.some((call) => {
    if (call.toolName !== "createEvent") return false;
    const output = call.result as { created?: boolean };
    return Boolean(output?.created);
  });
};

const evaluateTrajectory = (
  trajectory: TraceToolCall[],
  expected: EvalExpected,
  prompt: string
): TrajectoryEval => {
  const failedChecks: string[] = [];
  const checkResults: Record<string, boolean> = {};
  const toolSequence = trajectory.map((call) => call.toolName);

  for (const requiredTool of expected.requiredTools ?? []) {
    const ok = toolSequence.includes(requiredTool);
    checkResults["required:" + requiredTool] = ok;
    if (!ok) failedChecks.push("required:" + requiredTool);
  }

  for (const forbiddenTool of expected.forbiddenTools ?? []) {
    const ok = !toolSequence.includes(forbiddenTool);
    checkResults["forbidden:" + forbiddenTool] = ok;
    if (!ok) failedChecks.push("forbidden:" + forbiddenTool);
  }

  if (typeof expected.shouldCreateEvent === "boolean") {
    const created = wasCreateEventCreated(trajectory);
    const ok = created === expected.shouldCreateEvent;
    checkResults["created_event"] = ok;
    if (!ok) {
      failedChecks.push("created_event:" + String(expected.shouldCreateEvent));
    }
  }

  if (expected.ambiguityPolicy === "resolve-no-create") {
    const resolved = toolSequence.includes("resolveContact");
    checkResults["ambiguity:resolveContact"] = resolved;
    if (!resolved) failedChecks.push("ambiguity:missing_resolveContact");

    const ok = !wasCreateEventCreated(trajectory);
    checkResults["ambiguity:no_create_before_disambiguation"] = ok;
    if (!ok) failedChecks.push("ambiguity:created_event_before_disambiguation");

    checkResults["ambiguity:prompt_contains_ambiguous_contact"] =
      isAmbiguousPrompt(prompt);
  }

  return { failedChecks, checkResults, toolSequence };
};

const experimentExamples = examples.map((example) => ({
  input: { id: example.id, prompt: example.input },
  output: toExpected(example),
  metadata: {
    category:
      example.id === "ex-8"
        ? "negative-availability"
        : isAmbiguousPrompt(example.input)
          ? "ambiguous-contact"
          : "general",
    isAmbiguousContactCase: isAmbiguousPrompt(example.input),
  },
}));

const datasetName = "tool-loop-assistant-eval-v1";
const experimentName = "tool-loop-assistant-eval-" + new Date().toISOString();

const { datasetId } = await createOrGetDataset({
  name: datasetName,
  description:
    "Tool-loop personal assistant evaluation dataset (10 examples) for deterministic tool-use checks.",
  examples: experimentExamples,
});

console.log(
  "Prepared Phoenix dataset '" +
    datasetName +
    "' (id: " +
    datasetId +
    ") with " +
    experimentExamples.length +
    " examples."
);

const experimentResult = await runExperiment({
  dataset: { datasetId },
  experimentName,
  experimentDescription:
    "Task run for trajectory evals. Evaluations are executed in a trace-based second pass.",
  task: async (example) => {
    const input = example.input as Record<string, unknown>;
    const prompt = String(input.prompt);
    const run = await runAssistant(prompt);

    return {
      exampleId: String(input.id ?? ""),
      prompt,
      expectedSpec: (example.output ?? {}) as EvalExpected,
      responseText: run.text,
    } satisfies TaskOutput;
  },
});

const traceIdByExampleId = new Map<string, string>();
for (const run of Object.values(experimentResult.runs)) {
  const parsed = parseTaskOutput(run.output);
  if (run.traceId && parsed.exampleId) {
    traceIdByExampleId.set(parsed.exampleId, run.traceId);
  }
}

const expectedTraceIds = new Set(traceIdByExampleId.values());
const exampleIdByTraceId = new Map(
  Array.from(traceIdByExampleId.entries()).map(([exampleId, traceId]) => [
    traceId,
    exampleId,
  ])
);
const trajectoryByExampleId = new Map<string, TraceToolCall[]>();

const projectCandidates = Array.from(
  new Set(
    [experimentResult.projectName, phoenixProjectName]
      .map((name) => String(name ?? "").trim())
      .filter((name) => name.length > 0)
  )
);

const loadTrajectoriesFromSpans = async () => {
  trajectoryByExampleId.clear();

  for (const projectName of projectCandidates) {
    let cursor: string | null = null;

    while (true) {
      const page = await getSpans({
        client: phoenixClient,
        project: { projectName },
        limit: 200,
        cursor,
      });

      for (const span of page.spans) {
        if (span.name !== "ai.toolCall") continue;

        const traceId = span.context.trace_id;
        if (!expectedTraceIds.has(traceId)) continue;

        const exampleId = exampleIdByTraceId.get(traceId);
        if (!exampleId) continue;

        const attrs = span.attributes as Record<string, unknown>;
        const toolName = String(
          attrs["ai.toolCall.name"] ?? attrs["tool.name"] ?? ""
        );
        if (!toolName) continue;

        const rawResult = attrs["ai.toolCall.result"] ?? attrs["output.value"];
        const result = parseJsonIfString(rawResult);

        const existing = trajectoryByExampleId.get(exampleId) ?? [];
        existing.push({
          toolName,
          result,
          startTime: String((span as Record<string, unknown>).start_time ?? ""),
        });
        trajectoryByExampleId.set(exampleId, existing);
      }

      if (!page.nextCursor) break;
      cursor = page.nextCursor;
    }
  }

  for (const [exampleId, trajectory] of trajectoryByExampleId.entries()) {
    trajectory.sort((a, b) => a.startTime.localeCompare(b.startTime));
    trajectoryByExampleId.set(exampleId, trajectory);
  }
};

for (let attempt = 1; attempt <= 8; attempt++) {
  await loadTrajectoriesFromSpans();
  if (trajectoryByExampleId.size >= examples.length) break;
  await new Promise((resolve) => setTimeout(resolve, 1000));
}

console.log(
  "Trace trajectories reconstructed for " +
    trajectoryByExampleId.size +
    "/" +
    examples.length +
    " examples."
);

const evaluateFromTrace = (
  input: Record<string, unknown>,
  expected: EvalExpected
): TrajectoryEval => {
  const exampleId = String(input.id ?? "");
  const prompt = String(input.prompt ?? "");
  const trajectory = trajectoryByExampleId.get(exampleId) ?? [];

  if (trajectory.length === 0) {
    return {
      failedChecks: ["trace:missing_tool_spans"],
      checkResults: { "trace:found": false },
      toolSequence: [],
    };
  }

  return evaluateTrajectory(trajectory, expected, prompt);
};

const makeTraceEvaluator = ({
  name,
  passMessage,
  failMessagePrefix,
  failedPrefixes,
}: {
  name: string;
  passMessage: string;
  failMessagePrefix: string;
  failedPrefixes: string[];
}) => {
  return asExperimentEvaluator({
    name,
    kind: "CODE",
    evaluate: ({ input, expected }) => {
      const runInput = (input ?? {}) as Record<string, unknown>;
      const expectedSpec = (expected ?? {}) as EvalExpected;
      const evalResult = evaluateFromTrace(runInput, expectedSpec);

      const failed = evalResult.failedChecks.filter((check) =>
        failedPrefixes.some((prefix) => check.startsWith(prefix))
      );

      return {
        score: failed.length === 0 ? 1 : 0,
        label: failed.length === 0 ? "pass" : "fail",
        explanation:
          failed.length === 0
            ? passMessage
            : failMessagePrefix + failed.join(", "),
        metadata: {
          failedChecks: failed,
          toolSequence: evalResult.toolSequence,
        },
      };
    },
  });
};

const evaluators = [
  makeTraceEvaluator({
    name: "trace-required-tools-check",
    passMessage: "All required tools were found in trace-derived trajectory.",
    failMessagePrefix: "Missing required checks: ",
    failedPrefixes: ["required:", "trace:"],
  }),
  makeTraceEvaluator({
    name: "trace-forbidden-tools-check",
    passMessage: "No forbidden tools found in trace-derived trajectory.",
    failMessagePrefix: "Forbidden check failures: ",
    failedPrefixes: ["forbidden:", "trace:"],
  }),
  makeTraceEvaluator({
    name: "trace-create-event-check",
    passMessage: "Create-event behavior matches expectation from trace tool spans.",
    failMessagePrefix: "Create-event mismatch: ",
    failedPrefixes: ["created_event:", "trace:"],
  }),
  makeTraceEvaluator({
    name: "trace-ambiguity-disambiguation-check",
    passMessage: "Ambiguity policy passed using trace-derived trajectory.",
    failMessagePrefix: "Ambiguity violations: ",
    failedPrefixes: ["ambiguity:", "trace:"],
  }),
  asExperimentEvaluator({
    name: "trace-final-pass-check",
    kind: "CODE",
    evaluate: ({ input, expected }) => {
      const runInput = (input ?? {}) as Record<string, unknown>;
      const expectedSpec = (expected ?? {}) as EvalExpected;
      const evalResult = evaluateFromTrace(runInput, expectedSpec);

      const passed = evalResult.failedChecks.length === 0;
      return {
        score: passed ? 1 : 0,
        label: passed ? "pass" : "fail",
        explanation: passed
          ? "All deterministic checks passed on trace-derived trajectory."
          : "Failed checks: " + evalResult.failedChecks.join(", "),
        metadata: {
          failedChecks: evalResult.failedChecks,
          checkResults: evalResult.checkResults,
          toolSequence: evalResult.toolSequence,
        },
      };
    },
  }),
];

const evaluatedExperiment = await evaluateExperiment({
  experiment: experimentResult,
  evaluators,
});

const experimentUiUrl =
  phoenixCollectorEndpoint.replace(/\/$/, "") +
  "/datasets/" +
  datasetId +
  "/experiments";

console.log(
  "Phase 1 complete: task runs stored (experiment id: " +
    experimentResult.id +
    ")."
);
console.log(
  "Phase 2 complete: trace-based evaluations stored (evaluation rows: " +
    String(evaluatedExperiment.evaluationRuns?.length ?? 0) +
    ")."
);
console.log("Dataset: " + datasetName + " (id: " + datasetId + ")");
console.log("Experiment: " + experimentName);
console.log("View in Phoenix: " + experimentUiUrl);

## View Results in Phoenix

Open Phoenix to inspect:
- Project traces for this experiment run.
- Dataset `tool-loop-assistant-eval-v1`.
- Experiment evaluator columns prefixed with `trace-`.

What to verify:
- Evaluators were added by second-pass `evaluateExperiment`.
- Trajectories were reconstructed from `ai.toolCall` spans only (no app-level tool recording).
- Failing rows match trace-derived policy violations.

## Conclusion

This notebook demonstrates trace-native agent trajectory evaluation.

Key pattern:
1. Run experiment tasks first.
2. Reconstruct tool trajectory from OpenInference trace spans.
3. Run second-pass evaluators with `evaluateExperiment`.

This keeps evaluation logic decoupled from app internals and makes scoring reproducible from telemetry alone.